In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("../data/raw/APL_Logistics.csv", encoding="latin1")

print(df.shape)

(180519, 40)


# Data Cleaning

In [3]:
missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": round((df.isnull().sum()/len(df))*100,2)
})

missing[missing["Missing Values"] > 0].sort_values(
    by="Percentage",
    ascending=False
)

,Missing Values,Percentage
Customer Lname,8,0.0
Customer Zipcode,3,0.0


# Handle Missing Values

In [4]:
df["Customer Zipcode"].isnull().sum()

np.int64(3)

In [6]:
df["Customer Zipcode"] = df["Customer Zipcode"].fillna(
    df["Customer Zipcode"].mode()[0]
)

# Customer Lname

In [7]:
df["Customer Lname"] = df["Customer Lname"].fillna("Unknown")

In [8]:
df.isnull().sum().sort_values(
    ascending=False
).head(10)

Type                        0
Days for shipping (real)    0
Market                      0
Order City                  0
Order Country               0
Order Customer Id           0
Order Item Discount         0
Order Item Discount Rate    0
Order Item Product Price    0
Order Item Profit Ratio     0
dtype: int64

# Duplicate records 

In [9]:
df.duplicated().sum()

np.int64(0)

# Validate sales

In [10]:
(df["Sales"] <= 0).sum()

np.int64(0)

# Profit Validation

In [11]:
df["Order Profit Per Order"].describe()

count    180519.000000
mean         21.974989
std         104.433526
min       -4274.980000
25%           7.000000
50%          31.520000
75%          64.800000
max         911.800000
Name: Order Profit Per Order, dtype: float64

# Feature Engineering 

# customer full name 

In [13]:
df["Customer_Full_Name"] = (
    df["Customer Fname"].astype(str)
    + " "
    + df["Customer Lname"].astype(str)
)

# Profit margin %

In [14]:
df["Profit_Margin"] = (
    df["Order Profit Per Order"]
    /
    df["Sales"]
) * 100

In [15]:
df["Profit_Margin"].describe()

count    180519.000000
mean         10.832612
std          42.059373
min        -275.000000
25%           6.224000
50%          24.251213
75%          33.601434
max          50.044287
Name: Profit_Margin, dtype: float64

# Shipping delay

In [16]:
df["Shipping_Delay"] = (
    df["Days for shipping (real)"]
    -
    df["Days for shipment (scheduled)"]
)

In [17]:
df["Shipping_Delay"].describe()

count    180519.000000
mean          0.565807
std           1.490966
min          -2.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           4.000000
Name: Shipping_Delay, dtype: float64

# Discount Bucket

In [18]:
bins = [-1, 0.05, 0.10, 0.15, 0.20, 1]

labels = [
    "0-5%",
    "5-10%",
    "10-15%",
    "15-20%",
    "20%+"
]

df["Discount_Bucket"] = pd.cut(
    df["Order Item Discount Rate"],
    bins=bins,
    labels=labels
)

In [19]:
df["Discount_Bucket"].value_counts()

Discount_Bucket
0-5%      60171
5-10%     40116
15-20%    40116
10-15%    30087
20%+      10029
Name: count, dtype: int64

# Profitability Class (ML Target)

In [20]:
df["Profitability_Class"] = np.where(
    df["Order Profit Per Order"] > 0,
    1,
    0
)

In [21]:
df["Profitability_Class"].value_counts()

Profitability_Class
1    145558
0     34961
Name: count, dtype: int64

# Customer Value index

In [22]:
customer_profit = df.groupby(
    "Customer Id"
)["Order Profit Per Order"].transform("sum")

df["Customer_Value_Index"] = customer_profit

# Revenue Tier

In [23]:
df["Revenue_Tier"] = pd.qcut(
    df["Sales"],
    q=3,
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

In [24]:
df["Revenue_Tier"].value_counts()

Revenue_Tier
Low       70680
High      57209
Medium    52630
Name: count, dtype: int64

# Margin Category

In [25]:
conditions = [
    df["Profit_Margin"] < 0,
    df["Profit_Margin"] < 10,
    df["Profit_Margin"] < 20,
    df["Profit_Margin"] >= 20
]

choices = [
    "Loss Making",
    "Low Margin",
    "Medium Margin",
    "High Margin"
]

df["Margin_Category"] = np.select(
    conditions,
    choices,
    default="Unknown"
)

In [26]:
df["Margin_Category"].value_counts()

Margin_Category
High Margin      104077
Loss Making       33784
Low Margin        25481
Medium Margin     17177
Name: count, dtype: int64

# Data Validation 

# Final Shape

In [27]:
print(df.shape)

(180519, 48)


# New Features Created 

In [28]:
new_features = [
    "Customer_Full_Name",
    "Profit_Margin",
    "Shipping_Delay",
    "Discount_Bucket",
    "Profitability_Class",
    "Customer_Value_Index",
    "Revenue_Tier",
    "Margin_Category"
]

df[new_features].head()

,Customer_Full_Name,Profit_Margin,Shipping_Delay,Discount_Bucket,Profitability_Class,Customer_Value_Index,Revenue_Tier,Margin_Category
0,Richard Hernandez,31.941194,2,5-10%,1,159.69,High,High Margin
1,Mary Barrett,24.361090,0,15-20%,1,208.74,Medium,High Margin
2,Mary Barrett,43.682184,0,5-10%,1,208.74,Medium,High Margin
3,Mary Barrett,-20.946047,2,10-15%,0,208.74,Medium,Loss Making
4,Mary Barrett,20.000000,2,15-20%,1,208.74,Low,High Margin


# Save Clean Dataset

In [30]:
df.to_csv(
    "../data/processed/cleaned_data.csv",
    index=False
)

print("Dataset Saved Successfully")

Dataset Saved Successfully


# Data Cleaning & Feature Engineering Summary

### Cleaning Performed

- Missing values handled
- Duplicate records removed
- Invalid sales records removed
- Data quality validated

### New Features Created

1. Customer_Full_Name
2. Profit_Margin
3. Shipping_Delay
4. Discount_Bucket
5. Profitability_Class
6. Customer_Value_Index
7. Revenue_Tier
8. Margin_Category

### Business Benefits

These features will support:

- Customer profitability analysis
- Product profitability analysis
- Discount impact diagnostics
- Market performance evaluation
- Machine Learning profitability prediction